1) Instalación/imports

In [ ]:
!pip install ply

In [2]:
import sys
import ply.lex as lex
import ply.yacc as yacc
import json

# PLY intenta inspeccionar el archivo fuente del modulo __main__ para
# detectar reglas t_* duplicadas. En Jupyter, __main__ no tiene __file__,
# lo que hace que inspect.getsourcelines() falle con TypeError en vez de
# IOError (que es lo unico que PLY captura). Le seteamos un __file__ dummy
# para que ese chequeo falle de forma "esperada" y no rompa lex.lex().
if not hasattr(sys.modules["__main__"], "__file__"):
    sys.modules["__main__"].__file__ = "notebook.py"

2) Scanner

In [3]:
# ---------------------------------------
# Palabras reservadas y unidades
# ---------------------------------------

reserved = {
    "Recipe": "RECIPE",
    "Add": "ADD",
    "Mix": "MIX",
    "Bake": "BAKE",
    "Chill": "CHILL",
    "If": "IF",
    "then": "THEN",
    "else": "ELSE",
    "Serve": "SERVE",
    "minutes": "MINUTES",
    "portions": "PORTIONS",
    "g": "UNIT",
    "ml": "UNIT",
    "pcs": "UNIT"
}


# ---------------------------------------
# Lista de tokens
# ---------------------------------------

tokens = (
    "RECIPE",
    "ADD",
    "MIX",
    "BAKE",
    "CHILL",
    "IF",
    "THEN",
    "ELSE",
    "SERVE",
    "MINUTES",
    "PORTIONS",
    "ID",
    "INT",
    "STRING",
    "UNIT",
    "LLAVE_ABRE",
    "LLAVE_CIERRA",
    "PYC"
)


# ---------------------------------------
# Expresiones regulares simples
# ---------------------------------------

t_LLAVE_ABRE = r"\{"
t_LLAVE_CIERRA = r"\}"
t_PYC = r";"


# ---------------------------------------
# Caracteres ignorados
# ---------------------------------------

t_ignore = " \t\r"


# ---------------------------------------
# Token STRING
# Reconoce texto encerrado entre comillas dobles
# ---------------------------------------

def t_STRING(t):
    r'"[^"\n]*"'
    return t


# ---------------------------------------
# Token INT
# Reconoce uno o más dígitos
# ---------------------------------------

def t_INT(t):
    r"\d+"
    t.value = int(t.value)
    return t


# ---------------------------------------
# Token ID, palabras reservadas y unidades
# ---------------------------------------

def t_ID(t):
    r"[a-zA-Z][a-zA-Z0-9_]*"

    if t.value.endswith("_"):
        raise SyntaxError(f"Identificador inválido: {t.value}")

    if "__" in t.value:
        raise SyntaxError(f"Identificador inválido: {t.value}")

    t.type = reserved.get(t.value, "ID")
    return t


# ---------------------------------------
# Contador de líneas
# ---------------------------------------

def t_newline(t):
    r"\n+"
    t.lexer.lineno += len(t.value)


# ---------------------------------------
# Manejo de errores léxicos
# ---------------------------------------

def t_error(t):
    raise SyntaxError(
        f"Caracter no reconocido '{t.value[0]}' en la línea {t.lexer.lineno}"
    )


# ---------------------------------------
# Construcción del lexer
# ---------------------------------------

lexer = lex.lex()


# ---------------------------------------
# Función auxiliar para probar el scanner
# ---------------------------------------

def analizar_codigo(codigo):
    lexer.input(codigo)

    tokens_encontrados = []

    while True:
        tok = lexer.token()

        if not tok:
            break

        tokens_encontrados.append((tok.type, tok.value))

    return tokens_encontrados

3) Prueba del scanner

In [ ]:
programa = '''
Recipe "Tarta de manzana" {
    Add 200g harina;
    Add 100g azucar;
    Add 2pcs manzanas;
    Mix harina azucar manzanas;
    If tiene_horno then Bake 45 minutes else Chill 60 minutes;
    Serve 4 portions;
}
'''

resultado = analizar_codigo(programa)

for token, lexema in resultado:
    print(token, "->", lexema)

4. Contexto Semantico

In [5]:
# ---------------------------------------
# Tabla de símbolos semántica
# ---------------------------------------

class SemanticContext:

    def __init__(self):
        self.ingredients = set()

    def add_ingredient(self, name):
        if name in self.ingredients:
            raise Exception(f"Error semántico: ingrediente repetido '{name}'")

        self.ingredients.add(name)

    def validate_ingredients(self, ingredients):
        for ingredient in ingredients:
            if ingredient not in self.ingredients:
                raise Exception(
                    f"Error semántico: ingrediente no declarado '{ingredient}'"
                )


context = SemanticContext()


# ---------------------------------------
# Programa principal
# ---------------------------------------

def p_program(p):
    'program : RECIPE STRING LLAVE_ABRE steps LLAVE_CIERRA'
    recipe_name = p[2][1:-1]

    ingredients = []
    recipe_steps = []
    servings = None

    for step in p[4]:
        if step.get("type") == "ingredient":
            ingredients.append({
                "name": step["name"],
                "quantity": step["quantity"],
                "unit": step["unit"]
            })

        elif step.get("type") == "servings":
            servings = step["value"]

        else:
            recipe_steps.append(step)

    p[0] = {
        "recipe": recipe_name,
        "ingredients": ingredients,
        "steps": recipe_steps,
        "servings": servings
    }


# ---------------------------------------
# Lista de instrucciones
# ---------------------------------------

def p_steps_recursive(p):
    'steps : step steps'
    p[0] = [p[1]] + p[2]


def p_steps_single(p):
    'steps : step'
    p[0] = [p[1]]


def p_step(p):
    '''
    step : add_step
         | mix_step
         | bake_step
         | chill_step
         | conditional_step
         | serve_step
    '''
    p[0] = p[1]


# ---------------------------------------
# Instrucción Add
# ---------------------------------------

def p_add_step(p):
    'add_step : ADD INT UNIT ID PYC'

    if p[2] <= 0:
        raise Exception("Error semántico: la cantidad debe ser mayor que cero")

    context.add_ingredient(p[4])

    p[0] = {
        "type": "ingredient",
        "name": p[4],
        "quantity": p[2],
        "unit": p[3]
    }


# ---------------------------------------
# Instrucción Mix
# ---------------------------------------

def p_mix_step(p):
    'mix_step : MIX id_list PYC'

    context.validate_ingredients(p[2])

    p[0] = {
        "action": "mix",
        "ingredients": p[2]
    }


# ---------------------------------------
# Instrucción Bake
# ---------------------------------------

def p_bake_step(p):
    'bake_step : BAKE INT MINUTES PYC'

    if p[2] <= 0:
        raise Exception("Error semántico: el tiempo de cocción debe ser mayor que cero")

    p[0] = {
        "action": "bake",
        "time": p[2],
        "unit": "minutes"
    }


# ---------------------------------------
# Instrucción Chill
# ---------------------------------------

def p_chill_step(p):
    'chill_step : CHILL INT MINUTES PYC'

    if p[2] <= 0:
        raise Exception("Error semántico: el tiempo de enfriado debe ser mayor que cero")

    p[0] = {
        "action": "chill",
        "time": p[2],
        "unit": "minutes"
    }


# ---------------------------------------
# Instrucción Serve
# ---------------------------------------

def p_serve_step(p):
    'serve_step : SERVE INT PORTIONS PYC'

    if p[2] <= 0:
        raise Exception("Error semántico: las porciones deben ser mayores que cero")

    p[0] = {
        "type": "servings",
        "value": p[2]
    }


# ---------------------------------------
# Condicional
# ---------------------------------------

def p_conditional_step(p):
    'conditional_step : IF condition THEN action ELSE action PYC'

    p[0] = {
        "condition": p[2],
        "then": p[4],
        "else": p[6]
    }


def p_condition(p):
    'condition : ID'
    p[0] = p[1]


# ---------------------------------------
# Acciones dentro del condicional
# ---------------------------------------

def p_action_bake(p):
    'action : BAKE INT MINUTES'

    if p[2] <= 0:
        raise Exception("Error semántico: el tiempo de cocción debe ser mayor que cero")

    p[0] = {
        "action": "bake",
        "time": p[2],
        "unit": "minutes"
    }


def p_action_chill(p):
    'action : CHILL INT MINUTES'

    if p[2] <= 0:
        raise Exception("Error semántico: el tiempo de enfriado debe ser mayor que cero")

    p[0] = {
        "action": "chill",
        "time": p[2],
        "unit": "minutes"
    }


def p_action_mix(p):
    'action : MIX id_list'

    context.validate_ingredients(p[2])

    p[0] = {
        "action": "mix",
        "ingredients": p[2]
    }


# ---------------------------------------
# Lista de identificadores
# ---------------------------------------

def p_id_list_recursive(p):
    'id_list : ID id_list'
    p[0] = [p[1]] + p[2]


def p_id_list_single(p):
    'id_list : ID'
    p[0] = [p[1]]


# ---------------------------------------
# Manejo de errores sintácticos
# ---------------------------------------

def p_error(p):
    if p:
        raise SyntaxError(
            f"Error sintáctico en token {p.type} con valor {p.value}"
        )
    else:
        raise SyntaxError("Error sintáctico: fin de entrada inesperado")


# ---------------------------------------
# Construcción del parser
# ---------------------------------------

# write_tables=False y debug=False evitan que PLY intente escribir
# parsetab.py/parser.out a disco. Para eso necesita el archivo fuente
# de las funciones p_*, y en Jupyter inspect.getsourcefile(__main__)
# devuelve None (no hay un .py real), lo que rompía con
# TypeError: expected str, bytes or os.PathLike object, not NoneType.
parser = yacc.yacc(write_tables=False, debug=False)


# ---------------------------------------
# Función de traducción
# ---------------------------------------

def traducir_a_json(codigo):
    global context

    context = SemanticContext()

    resultado = parser.parse(codigo, lexer=lexer)

    return json.dumps(resultado, indent=2, ensure_ascii=False)

6. Funcion principal de traduccion

In [ ]:
programa = '''
Recipe "Tarta de manzana" {
    Add 200g harina;
    Add 100g azucar;
    Add 2pcs manzanas;
    Mix harina azucar manzanas;
    If tiene_horno then Bake 45 minutes else Chill 60 minutes;
    Serve 4 portions;
}
'''

print(traducir_a_json(programa))